In [1]:
import warnings
import os 
import subprocess 

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
#import scienceplots  # noqa: F401
import torch
import torch.distributions as D

# flow_matching
from flow_matching.solver import ODESolver
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# SKLearn
from sklearn.svm import SVC

from data import generate_quad_gmm
from model import vf, prior, y_null_val, ConditionedVelocityModelWrapper

torch.manual_seed(42)
device = torch.device("cpu")

In [ ]:
# 1. Define the folder and filename
folder_path = './checkpoints/' # Change this to your desired folder
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

checkpoint_path = os.path.join(folder_path, 'cond_fm.pth')
ckpt = torch.load(checkpoint_path)
vf.load_state_dict(ckpt["vf_state_dict"])
prior.load_state_dict(ckpt["prior_state_dict"])

In [3]:
x_1, y, d, _ = generate_quad_gmm(1000)
gaussian_log_density = D.Independent(
    D.Normal(torch.zeros(2, device=device), torch.ones(2, device=device)), 1
).log_prob

In [4]:
omegas = np.linspace(0, 1, 10)
n_steps = 50  # must match your time_grid

idx = np.arange(len(y))
y_np = y.detach().cpu().numpy() if hasattr(y, "detach") else np.asarray(y)
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=y_np
)

omega_means, omega_stds = [], []  # one (mean/std) series per omega

for omega in omegas:
    wrapped_vf = ConditionedVelocityModelWrapper(
        velocity_model=vf, cfg_scale=omega
    )
    solver = ODESolver(
        velocity_model=wrapped_vf
    )

    out = solver.compute_likelihood(
        x_1=x_1,
        y=y,
        time_grid=torch.linspace(1, 0, n_steps),
        method="midpoint",
        step_size=None,
        exact_divergence=False,
        log_p0=gaussian_log_density,
        return_intermediates=True,
    )

    series_mean = np.empty(len(out[0]), dtype=float)
    series_std = np.empty(len(out[0]), dtype=float)

    for t, X_t in enumerate(out[0]):
        Xt = X_t.detach().cpu().numpy() if hasattr(X_t, "detach") else np.asarray(X_t)

        X_train, X_test = Xt[train_idx], Xt[test_idx]
        y_train, y_test = y_np[train_idx], y_np[test_idx]

        clf = SVC(kernel="rbf", probability=True, random_state=42)
        clf.fit(X_train, y_train)

        lp = clf.predict_log_proba(X_test)  # (n_test, n_classes)

        # True-class log-probs
        col_for_class = {c: j for j, c in enumerate(clf.classes_)}
        cols = np.array([col_for_class[yi] for yi in y_test])
        true_logps = lp[np.arange(len(y_test)), cols]

        infos = true_logps - np.log(1 / 4)

        series_mean[t] = infos.mean()
        series_std[t] = infos.std()  # std across test samples at timestep t

    omega_means.append(series_mean)
    omega_stds.append(series_std)

# shape: (len(omegas), n_steps); reverse time to match your convention
omega_means = np.stack(omega_means)[:, ::-1]
omega_stds = np.stack(omega_stds)[:, ::-1]

In [ ]:
# --- Setup Data ---
# (Assuming omegas, omega_means, and omega_stds are defined)
ts = np.linspace(0, 1, 50)
cmap = mpl.cm.viridis

# 1. Define the normalization based on the range of omega values
norm = mpl.colors.Normalize(vmin=omegas.min(), vmax=omegas.max())
colors = cmap(norm(omegas))

# 2. Initialize the plot
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

# 3. Plot the datar
for k in range(len(omega_means)):
    ax.plot(ts, omega_means[k], color=colors[k])

# 4. Reference lines and labels
ax.set_xlabel("t", fontsize=24)
ax.set_ylabel(r"$\hat{I}(X_t;y)$ / nats", fontsize=24)
ax.set_xlim(0,1)
ax.axhline(-np.log(1 / 4), ls="--", c="black", label=r"$I(X_t;y)_{\text{max}}$")
ax.axhline(0, ls="-.", c="black", label=r"$I(X_t;y)_{\text{min}}$")


# 5. Add the Colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.03, aspect=40)
cbar.set_label(r"$\omega$", fontsize=24, rotation=0)

# 6. Final touches
ax.legend(loc="center right", fontsize=18)
plt.tight_layout()
plt.show()